<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_12_Real_Source_TRACE_Athena_Recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import os, sys, json, re, hashlib, csv, zipfile, io, platform, shutil
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from collections import Counter
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
BASE = Path('/content') if Path('/content').exists() else Path.cwd()
MYDRIVE = BASE / 'drive/MyDrive'
PROJECT = Path(os.environ.get('ICHI_PROJECT_DIR', str(MYDRIVE / 'NeuroFHIR_ICHI2027')))
OUT = PROJECT / 'phase12_outputs'
OUT.mkdir(parents=True, exist_ok=True)
PRIVATE = PROJECT / 'phase12_private'
PRIVATE.mkdir(parents=True, exist_ok=True)
REPORT = OUT / 'ICHI2027_Phase12_SHAREABLE.json'
SOURCE_NAME = 'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'
PRIOR_NAME = 'ICHI2027_Phase11_SHAREABLE.json'
STARTED = datetime.now(timezone.utc).isoformat()

FIELD_MAP = {
    'LHIPPOC':'ST29SV','RHIPPOC':'ST88SV',
    'LENTORHIN':'ST24CV','RENTORHIN':'ST83CV',
    'LMIDTEMP':'ST40CV','RMIDTEMP':'ST99CV','ICV':'ST10CV',
}
ALL_FEATURES = tuple(FIELD_MAP) + ('VENTRICLES',)

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4<<20),b''):
            h.update(chunk)
    return h.hexdigest()

def locate(name):
    roots=[PROJECT/'phase11_outputs',PROJECT/'source_data',PROJECT/'inputs',PROJECT/'source_exports',BASE,BASE/'inputs']
    return next((r/name for r in roots if (r/name).is_file()),None)

def dec(x):
    try:
        d=Decimal(str(x).strip())
        return d if d.is_finite() else None
    except (InvalidOperation,ValueError):
        return None

def norm_date(s):
    return pd.to_datetime(s,errors='coerce',format='mixed').dt.strftime('%Y-%m-%d').fillna('')

def one_line(err):
    return type(err).__name__


Mounted at /content/drive


In [2]:
prior_path=locate(PRIOR_NAME)
if prior_path is None and IN_COLAB:
    from google.colab import files
    upload=files.upload()
    if PRIOR_NAME in upload:
        prior_path=PRIVATE/PRIOR_NAME
        prior_path.write_bytes(upload[PRIOR_NAME])
if prior_path is None:
    raise FileNotFoundError(PRIOR_NAME)
prior=json.loads(prior_path.read_text(encoding='utf-8'))
if prior.get('phase')!='11_exact_ADNI_export_crosswalk_and_athena_gate':
    raise ValueError('Wrong Phase 11 prerequisite JSON')
prior_sha=digest(prior_path)

source_path=locate(SOURCE_NAME)
if source_path is None and IN_COLAB:
    from google.colab import files
    upload=files.upload()
    if SOURCE_NAME in upload:
        source_path=PRIVATE/SOURCE_NAME
        source_path.write_bytes(upload[SOURCE_NAME])
if source_path is None:
    raise FileNotFoundError(SOURCE_NAME)
source=pd.read_csv(source_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
source.columns=source.columns.str.strip()
expected={'RID','PTID'}|{f'{t}_{f}' for t in ('BL','M12')
    for f in ('EXAMDATE','IMAGEUID','LONIUID','VERSION')+ALL_FEATURES}
missing=sorted(expected-set(source.columns))
if missing: raise ValueError('Real source schema changed; missing: '+repr(missing))
if len(source)!=prior['source_profile']['rows'] or len(source.columns)!=prior['source_profile']['columns']:
    raise ValueError('Source size/schema changed since Phase 11')
source['RID']=source.RID.str.strip()
if source.RID.eq('').any() or source.RID.duplicated().any():
    raise ValueError('Source RID missing or duplicated')
source_sha=digest(source_path)
(PRIVATE/'private_run_manifest.json').write_text(json.dumps({
    'source_sha256':source_sha,'phase11_sha256':prior_sha,'run_utc':STARTED},indent=2))


226

In [3]:
SCAN_ROOTS=[PROJECT,BASE/'inputs',BASE/'source_metadata',BASE/'trace_outputs',
    MYDRIVE/'NeuroFHIR_TRACE',MYDRIVE/'NeuroFHIR-TRACE',MYDRIVE/'TRACE',
    MYDRIVE/'ADNI',MYDRIVE/'Colab Notebooks']
SCAN_ROOTS=[p for p in SCAN_ROOTS if p.is_dir()]
ALLOWED={'.csv','.tsv','.parquet','.rda','.rdata','.zip','.json','.ipynb','.xlsx','.pdf'}
NEEDLE=re.compile(r'ucsffsl|freesurfer|longitudinal|source.?export|adni|athena|vocab|trace|reliability|calibration|19b|20b|23b|24a|24b',re.I)
SKIP_DIR={'node_modules','.git','__pycache__','.ipynb_checkpoints'}
seen=set(); candidates=[]; filenames_scanned=0; scan_truncated=False
for root in SCAN_ROOTS:
    for dirpath, dirs, names in os.walk(root):
        dirs[:]=[d for d in dirs if d not in SKIP_DIR]
        if '/phase' in str(dirpath).replace('\\','/') and 'outputs' in str(dirpath):
            dirs.clear(); continue
        for name in names:
            filenames_scanned+=1
            if filenames_scanned > 75000:
                scan_truncated=True; break
            p=Path(dirpath)/name
            if p in seen or p.suffix.lower() not in ALLOWED or not NEEDLE.search(name):
                continue
            seen.add(p)
            try: size=p.stat().st_size
            except OSError: continue
            if size>3_000_000_000: continue
            lname=name.lower()
            if p.suffix.lower()=='.zip' and ('athena' in lname or 'vocab' in lname): kind='athena'
            elif p.suffix.lower() in {'.csv','.tsv','.rda','.rdata','.parquet'} and ('ucsffsl' in lname or ('raw' in lname and 'adni' in lname)): kind='raw_source'
            elif p.suffix.lower() in {'.csv','.tsv','.parquet','.json'} and ('trace' in lname or 'reliab' in lname or 'calib' in lname): kind='trace'
            elif p.suffix.lower()=='.ipynb' and ('19b' in lname or '20b' in lname or '23b' in lname or '24' in lname or 'trace' in lname): kind='prior_notebook'
            elif p.suffix.lower() in {'.pdf','.xlsx','.json'} and ('adni' in lname or 'source' in lname or 'ucsffsl' in lname): kind='source_document'
            else: continue
            candidates.append({'path':p,'kind':kind,'bytes':size})
        if scan_truncated: break
    if scan_truncated: break
for p in BASE.iterdir():
    if not p.is_file() or p in seen or p.suffix.lower() not in ALLOWED or not NEEDLE.search(p.name):
        continue
    seen.add(p)
    lower=p.name.lower()
    if p.suffix.lower()=='.zip' and ('vocab' in lower or 'athena' in lower):kind='athena'
    elif p.suffix.lower() in {'.csv','.tsv','.rda','.rdata','.parquet'} and 'ucsffsl' in lower:kind='raw_source'
    elif p.suffix.lower() in {'.csv','.tsv','.parquet','.json'} and ('trace' in lower or 'reliab' in lower):kind='trace'
    else:continue
    candidates.append({'path':p,'kind':kind,'bytes':p.stat().st_size})
counts=dict(Counter(x['kind'] for x in candidates))
(PRIVATE/'candidate_inventory_PRIVATE.json').write_text(json.dumps({
    'filenames_scanned':filenames_scanned,'truncated':scan_truncated,
    'files':[{'path':str(x['path']),'kind':x['kind'],'bytes':x['bytes']} for x in candidates]
},indent=2))


4721

In [4]:
raw_options=sorted((x for x in candidates if x['kind']=='raw_source'),
    key=lambda x:('ucsffsl51' not in x['path'].name.lower(),-x['bytes'],str(x['path'])))
raw_scan=[]; raw_datasets=[]
for entry in raw_options[:25]:
    p=entry['path']; profile={'type':p.suffix.lower(),'bytes':entry['bytes'],'readable':False,
                             'table_signature_present':False,'error':None}
    try:
        if p.suffix.lower() in {'.csv','.tsv'}:
            with open(p,encoding='utf-8-sig',errors='replace',newline='') as f:
                header=f.readline()
            sep='\t' if '\t' in header else ','
            hdr=[v.strip() for v in next(csv.reader([header],delimiter=sep))]
            if not {'RID','EXAMDATE','IMAGEUID','LONIUID'}.issubset(hdr) or not (set(FIELD_MAP.values())&set(hdr)):
                profile['missing_required_headers']=True
                raw_scan.append(profile); continue
            raw=pd.read_csv(p,dtype=str,keep_default_na=False,encoding='utf-8-sig',sep=sep,low_memory=False)
            tables=[('single',raw)]
        elif p.suffix.lower() in {'.rda','.rdata'}:
            import importlib.util
            if importlib.util.find_spec('pyreadr') is None:
                import subprocess
                subprocess.run([sys.executable,'-m','pip','-q','install','pyreadr'],check=True)
            import pyreadr
            tables=list(pyreadr.read_r(str(p)).items())
        elif p.suffix.lower()=='.parquet':
            raw=pd.read_parquet(p)
            tables=[('single',raw)]
        else: continue
        for object_name,raw in tables:
            if not isinstance(raw,pd.DataFrame): continue
            raw.columns=[str(c).strip() for c in raw.columns]
            keys={'RID','EXAMDATE','IMAGEUID','LONIUID'}
            fields=set(FIELD_MAP.values())&set(raw.columns)
            if not keys.issubset(raw.columns) or not fields: continue
            raw=raw.astype(str).replace({'nan':'','None':'','NaT':''})
            raw_datasets.append({'path':p,'object_name':str(object_name),'df':raw,'fields':sorted(fields)})
            profile['readable']=True
            profile['table_signature_present']=True
            profile['source_columns']=len(raw.columns)
            profile['feature_fields_present']=len(fields)
            profile['rows']=len(raw)
        if not profile['readable']:profile['error']='NO_MATCHING_TABLE_OBJECT'
    except Exception as e:
        profile['error']=one_line(e)
    raw_scan.append(profile)


In [5]:
events=[]
for t in ('BL','M12'):
    frame=source[['RID',f'{t}_EXAMDATE',f'{t}_IMAGEUID',f'{t}_LONIUID',f'{t}_VERSION']+
                  [f'{t}_{f}' for f in ALL_FEATURES]].copy()
    frame.columns=['RID','EXAMDATE','IMAGEUID','LONIUID','VERSION']+list(ALL_FEATURES)
    frame['EXAMDATE']=norm_date(frame.EXAMDATE)
    for k in ('RID','IMAGEUID','LONIUID','VERSION'):
        frame[k]=frame[k].str.strip()
    frame['TIMEPOINT']=t
    events.append(frame)
local_events=pd.concat(events,ignore_index=True)

reconciliations=[]
for obj in raw_datasets:
    raw=obj['df'].copy()
    raw['EXAMDATE']=norm_date(raw.EXAMDATE)
    for k in ('RID','IMAGEUID','LONIUID','VERSION'):
        if k in raw.columns:raw[k]=raw[k].astype(str).str.strip()
    keys=['RID','EXAMDATE','IMAGEUID','LONIUID']
    if 'VERSION' in raw.columns:keys.append('VERSION')
    raw_dups=int(raw.duplicated(keys,keep=False).sum())
    raw_unique=raw.loc[~raw.duplicated(keys,keep=False)]
    merged=local_events.merge(raw_unique,how='left',on=keys,validate='many_to_one',indicator=True,suffixes=('','_ORIGINAL'))
    matched=merged['_merge'].eq('both')
    per_field={}
    for field,old in FIELD_MAP.items():
        rec={'original_field':old,'present_in_raw':old in raw.columns,'compared':0,
             'exact_decimal_matches':0,'different':0,'nonnumeric_or_blank':0}
        if old in raw.columns:
            pairs=merged.loc[matched,[field,old]]
            for a,b in pairs.itertuples(index=False,name=None):
                v1,v2=dec(a),dec(b)
                if v1 is None or v2 is None:rec['nonnumeric_or_blank']+=1
                else:
                    rec['compared']+=1
                    if v1==v2:rec['exact_decimal_matches']+=1
                    else:rec['different']+=1
        per_field[field]=rec
    counts_by_tp={t:int((matched & merged.TIMEPOINT.eq(t)).sum()) for t in ('BL','M12')}
    report={
        'matched_events':int(matched.sum()),'total_events':int(len(local_events)),
        'matched_by_timepoint':counts_by_tp,'ambiguous_raw_rows_excluded':raw_dups,
        'raw_has_LONISID':'LONISID' in raw.columns,
        'raw_has_VERSION':'VERSION' in raw.columns,
        'field_comparison':per_field,
        'full_exact_reconciliation':bool(matched.all() and raw_dups==0 and
            all(r['present_in_raw'] and r['compared']==len(local_events) and
                r['different']==0 for r in per_field.values())),
    }
    if 'LONISID' in raw.columns:
        sid=merged.loc[matched,'LONISID'].fillna('').astype(str).str.strip()
        report['matched_rows_with_original_LONISID']=int(sid.ne('').sum())
    reconciliations.append((obj,report))

best_obj,best_report=(max(reconciliations,key=lambda x:(x[1]['matched_events'],
                     sum(r['exact_decimal_matches'] for r in x[1]['field_comparison'].values())))
                     if reconciliations else (None,None))
if best_obj is not None:
    (PRIVATE/'source_provenance_PRIVATE.json').write_text(json.dumps({
        'original_raw_sha256':digest(best_obj['path']),
        'original_object_name':best_obj['object_name'],
        'source_export_sha256':source_sha,'reconciliation':best_report},indent=2))


In [6]:
trace_candidates=sorted((x for x in candidates if x['kind']=='trace' and
   not x['path'].name.startswith('TRACE_ADNI1_') and
   not x['path'].name.startswith('ICHI2027_Phase')),key=lambda x:str(x['path']))
trace_profiles=[]
for e in trace_candidates[:60]:
    p=e['path']; profile={'file_type':p.suffix.lower(),'readable':False,
                         'join_key_present':False,'state_present':False,
                         'rule_version_present':False,'calibration_reference_present':False,
                         'exact_image_pair_keys_present':False,'original_provenance_approved':False}
    try:
        if p.suffix.lower()=='.parquet':
            import pyarrow.parquet as pq
            cols=pq.ParquetFile(p).schema.names
        elif p.suffix.lower() in {'.csv','.tsv'}:
            with open(p,encoding='utf-8-sig',errors='replace') as f:
                line=f.readline()
            cols=[x.strip() for x in next(csv.reader([line],delimiter='\t' if '\t' in line else ','))]
        elif p.suffix.lower()=='.json' and e['bytes']<15_000_000:
            data=json.loads(p.read_text(encoding='utf-8'))
            if isinstance(data,dict) and 'records' in data and isinstance(data['records'],list):
                cols=list(data['records'][0]) if data['records'] else []
            elif isinstance(data,list): cols=list(data[0]) if data else []
            elif isinstance(data,dict): cols=list(data)
            else: cols=[]
        else:continue
        low={c.lower() for c in cols}
        profile['readable']=True
        profile['column_count']=len(cols)
        profile['join_key_present']=bool(low & {'rid','ptid','participant_id'})
        profile['state_present']=any(('state' in c or 'status' in c) and
            ('trace' in c or 'reliab' in c) for c in low)
        profile['rule_version_present']=any('rule' in c and ('version' in c or 'id' in c) for c in low)
        profile['calibration_reference_present']=any('calibration' in c or 'reference' in c for c in low)
        profile['exact_image_pair_keys_present']={'bl_imageuid','m12_imageuid'}.issubset(low)
    except Exception as ex: profile['error_type']=one_line(ex)
    trace_profiles.append(profile)


In [7]:
archive_candidates=sorted((x for x in candidates if x['kind']=='athena'),key=lambda x:-x['bytes'])
required={'CONCEPT.CSV','VOCABULARY.CSV','DOMAIN.CSV','CONCEPT_CLASS.CSV',
          'CONCEPT_RELATIONSHIP.CSV','CONCEPT_ANCESTOR.CSV','CONCEPT_SYNONYM.CSV',
          'RELATIONSHIP.CSV','DRUG_STRENGTH.CSV'}
patterns={
 'mri':r'\bmagnetic resonance\b|\bmri\b',
 'hippocampus':r'hippocamp', 'entorhinal':r'entorhinal',
 'middle_temporal':r'middle temporal','ventricles':r'ventric',
 'intracranial':r'intracranial',
 'cubic_millimeter':r'cubic millimet|millimeter cubed|mm\^?3',
}
athena={'snapshot_present':False,'full_package_present':False,
        'candidate_archive_count':len(archive_candidates),'scanned_concept_rows':0,
        'vocabulary_entries':0,'concept_candidates_by_family':{},
        'snapshot_digest_recorded_privately':False,'manual_review_required':True}
for entry in archive_candidates[:12]:
    try:
        p=entry['path']
        with zipfile.ZipFile(p) as z:
            names={Path(n).name.upper():n for n in z.namelist() if not n.endswith('/')}
            if not {'CONCEPT.CSV','VOCABULARY.CSV'}.issubset(names):continue
            full=required.issubset(names)
            with z.open(names['CONCEPT.CSV']) as fd:
                wrapper=io.TextIOWrapper(fd,encoding='utf-8-sig',errors='replace',newline='')
                reader=csv.DictReader(wrapper,delimiter='\t')
                if not {'concept_id','concept_name','standard_concept','invalid_reason','domain_id','vocabulary_id'}.issubset(reader.fieldnames or []):
                    raise ValueError('Not a complete Athena CONCEPT.csv')
                counts=Counter(); extracted=[]; rows_scanned=0
                for row in reader:
                    rows_scanned+=1
                    if row.get('standard_concept','').strip()!='S' or row.get('invalid_reason','').strip():continue
                    for key,pat in patterns.items():
                        if re.search(pat,row.get('concept_name',''),re.I):
                            counts[key]+=1
                            if counts[key]<=25:
                                extracted.append({'family':key,'concept_id':row['concept_id'],
                                    'concept_name':row['concept_name'],'domain_id':row['domain_id'],
                                    'vocabulary_id':row['vocabulary_id'],
                                    'valid_end_date':row.get('valid_end_date')})
            with z.open(names['VOCABULARY.CSV']) as fd:
                wrapper=io.TextIOWrapper(fd,encoding='utf-8-sig',errors='replace',newline='')
                versions=list(csv.DictReader(wrapper,delimiter='\t'))
            digest_val=digest(p)
            (PRIVATE/'athena_candidates_PRIVATE.json').write_text(json.dumps({
                'archive_sha256':digest_val,'vocabularies':versions,
                'not_approved_concept_candidates':extracted},indent=2))
            athena.update({'snapshot_present':True,'full_package_present':full,
                'scanned_concept_rows':rows_scanned,'vocabulary_entries':len(versions),
                'concept_candidates_by_family':dict(counts),
                'snapshot_digest_recorded_privately':True})
            break
    except Exception as ex:
        athena.setdefault('read_errors',[]).append(one_line(ex))


In [8]:
notebooks=sorted((x for x in candidates if x['kind']=='prior_notebook'),
                key=lambda x:str(x['path']))
prior_notebook_evidence=[]
for entry in notebooks[:35]:
    if entry['bytes']>30_000_000:continue
    try:
        nb=json.loads(entry['path'].read_text(encoding='utf-8'))
        blocks=[''.join(c.get('source',[])) if isinstance(c.get('source',[]),list)
                else c.get('source','') for c in nb.get('cells',[]) if c.get('cell_type')=='code']
        code='\n'.join(blocks)
        named_artifacts=sorted(set(re.findall(r'\b[A-Za-z0-9_-]*(?:trace|reliab|calib)[A-Za-z0-9_-]*\.(?:csv|parquet|json)\b',code,flags=re.I)))
        if named_artifacts or re.search(r'REVIEW_REQUIRED|TRUSTED|CAUTION',code):
            prior_notebook_evidence.append({
                'contains_TRACE_state_code':bool(re.search(r'REVIEW_REQUIRED|TRUSTED|CAUTION',code)),
                'candidate_output_file_count':len(named_artifacts),
                'notebook_digest_recorded_privately':True})
    except Exception:continue


In [9]:
source_reconciled=bool(best_report and best_report['full_exact_reconciliation'])
public=prior.get('public_adni_source_reference',{})
public_features=public.get('source_feature_candidates',{})
field_status={}
for f,raw_field in FIELD_MAP.items():
    evidence=best_report['field_comparison'].get(f,{}) if best_report else {}
    documented=public_features.get(f,{})
    exact=bool(best_report and evidence.get('compared')==len(local_events) and evidence.get('different')==0)
    field_status[f]={
        'mapped_raw_field_candidate':raw_field,
        'full_local_raw_numeric_match':exact,
        'public_source_dictionary_unit':documented.get('public_unit_candidate'),
        'eligible_for_unit_provenance_review':bool(exact and documented.get('public_unit_candidate')=='mm3'),
        'automatically_approved':False,
    }

out={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'12_original_source_and_TRACE_vocabulary_recovery',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'status':'REAL_RAW_SOURCE_EVIDENCE_CROSSCHECK_ONLY_NO_NEW_MICDM_LOAD_OR_FOUR_ARM_COMPARISON',
 'privacy':'No ADNI participant records, identifiers, source file paths, raw tables, or TRACE observations exported.',
 'phase11_shareable_sha256':prior_sha,
 'runtime':{'python':platform.python_version(),'pandas':pd.__version__,
     'pyreadr':getattr(sys.modules.get('pyreadr'), '__version__',None)},
 'local_verified_source_rows':len(source),
 'drive_discovery':{'filenames_scanned':filenames_scanned,'truncated':scan_truncated,
                    'categorized_candidates':counts},
 'original_raw_export':{
   'candidate_files_inspected':len(raw_scan),
   'tables_with_required_raw_key_fields':len(raw_datasets),
   'best_matching_raw_table_found':best_report is not None,
   'best_table_match_summary':best_report,
   'full_exact_reconciliation':source_reconciled,
   'candidate_field_unit_review':field_status,
   'ventricles_aggregate_definition_confirmed':False,
 },
 'trace_artifacts':{'candidate_files_inspected':len(trace_profiles),
   'files_with_state_rule_reference_and_join_key':sum(x['join_key_present'] and x['state_present'] and
       x['rule_version_present'] and x['calibration_reference_present'] for x in trace_profiles),
   'files_with_pairwise_image_lineage':sum(x['exact_image_pair_keys_present'] for x in trace_profiles),
   'original_trace_provenance_approved':False},
 'prior_trace_notebook_inventory':{'relevant_notebooks_found':len(prior_notebook_evidence),
       'potential_reliability_logic_notebooks':sum(x['contains_TRACE_state_code'] for x in prior_notebook_evidence)},
 'athena':athena,
 'gates':{
    'exact_original_raw_export_reconciled':source_reconciled,
    'exact_local_export_units_approved':False,
    'dicom_study_and_series_UIDs_verified':False,
    'original_TRACE_reliability_state_calibration_approved':False,
    'athena_full_snapshot_loaded':athena['full_package_present'],
    'OMOP_concept_mappings_approved':False,
    'official_MICDM_load_completed':False,
    'genuine_four_arm_experiment_completed':False,
 },
 'actual_comparator_scores':None,
 'next_required_evidence':[],
}
if not raw_datasets:out['next_required_evidence'].append('ORIGINAL_UCSFFSL51_RAW_EXPORT_FROM_AUTHORIZED_ADNI_DOWNLOAD')
if not source_reconciled and raw_datasets:out['next_required_evidence'].append('RESOLVE_SOURCE_RAW_KEY_AND_MEASUREMENT_DISCREPANCIES')
if not athena['full_package_present']:out['next_required_evidence'].append('FULL_ATHENA_VOCABULARY_ZIP')
if out['trace_artifacts']['files_with_state_rule_reference_and_join_key']==0:
    out['next_required_evidence'].append('ORIGINAL_TRACE_SIDECAR_WITH_STATE_RULE_VERSION_AND_CALIBRATION_REFERENCE')
out['next_required_evidence'].append('REVIEW_APPROVE_SOURCE_UNITS_DICOM_LINKAGE_AND_OMOP_CONCEPT_MAP_BEFORE_TARGET_LOAD')
REPORT.write_text(json.dumps(out,indent=2),encoding='utf-8')
print(json.dumps({'phase':out['phase'],'found_original_table':bool(raw_datasets),
  'exact_reconciliation':source_reconciled,'athena_full_snapshot':athena['full_package_present'],
  'potential_TRACE_sidecars':out['trace_artifacts']['files_with_state_rule_reference_and_join_key'],
  'next_required_evidence':out['next_required_evidence']},indent=2))


{
  "phase": "12_original_source_and_TRACE_vocabulary_recovery",
  "found_original_table": false,
  "exact_reconciliation": false,
  "athena_full_snapshot": false,
  "potential_TRACE_sidecars": 0,
  "next_required_evidence": [
    "ORIGINAL_UCSFFSL51_RAW_EXPORT_FROM_AUTHORIZED_ADNI_DOWNLOAD",
    "FULL_ATHENA_VOCABULARY_ZIP",
    "ORIGINAL_TRACE_SIDECAR_WITH_STATE_RULE_VERSION_AND_CALIBRATION_REFERENCE",
    "REVIEW_APPROVE_SOURCE_UNITS_DICOM_LINKAGE_AND_OMOP_CONCEPT_MAP_BEFORE_TARGET_LOAD"
  ]
}


In [10]:
if IN_COLAB:
    from google.colab import files
    files.download(str(REPORT))
else:
    print(str(REPORT))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>